# Notebook 1 - Web Scraping (Playwright)

**Project:** Web Scraping and Product Price Analysis System  
**Source:** <https://scrapingsandbox.com/data-table> (a practice website made for scraping)

**Run order:** this notebook (1) -> `analysis/product_price_analysis.ipynb` (2) -> `dashboard/Product_Price_Analysis_Dashboard.ipynb` (3)

**What this notebook does**
1. Installs Playwright and the Chromium browser.
2. Opens the Data Table page and walks through 20 pages.
3. Reads every table row and cell into a Python list.
4. Builds a Pandas DataFrame, validates it, cleans it.
5. Saves `scraped_product_data.csv` and `scraped_product_data.xlsx` into `data/`.

> **Status note (please read):** the page selectors and the "Next" button logic in Section 3 were
> written from the project description and have **not been verified against the live website**.
> If your own working scraper differs, paste your logic into Section 3 and keep the rest.

## 1. Install Playwright and the Chromium browser

`pip install playwright` only installs the Python library. The browser itself must be installed separately with `playwright install`.

In [ ]:
# Run once per Colab session (takes about a minute).
!pip install -q playwright pandas openpyxl
!playwright install chromium
!playwright install-deps chromium

## 2. Imports and configuration

In [ ]:
import re
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

from pathlib import Path

def find_project_root():
    """Return the repository root whether the notebook runs from a sub-folder
    (local / cloned repo) or from /content (a fresh Google Colab session)."""
    cwd = Path.cwd()
    return cwd.parent if cwd.name in {"scraping", "analysis", "dashboard"} else cwd

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "scraped_product_data.csv"
XLSX_PATH = DATA_DIR / "scraped_product_data.xlsx"
print("Project root :", PROJECT_ROOT)
print("Data folder  :", DATA_DIR)

# ---- Scraping settings (project scope: do not change without updating the docs) ----
URL = "https://scrapingsandbox.com/data-table"
TOTAL_PAGES = 20

# The 10 columns expected in each table row, in order.
COLUMNS = ["ID", "Image", "Product_Name", "Vendor", "Category",
           "Price", "Rating", "Stock", "SKU", "Variants"]

# CSS selector for the table rows (ASSUMPTION - verify with your browser's Inspect tool).
ROW_SELECTOR = "table tbody tr"

## 3. Scrape all pages

Colab already runs an event loop, so the **async** Playwright API is used with `await`.

In [ ]:
async def read_current_page():
    """Read every row on the page that is currently shown. Returns a list of dicts."""
    rows = await page.query_selector_all(ROW_SELECTOR)
    page_records = []
    for row in rows:
        cells = await row.query_selector_all("td")
        if len(cells) < len(COLUMNS):
            continue  # skip header-like or malformed rows
        texts = [(await c.inner_text()).strip() for c in cells[:len(COLUMNS)]]
        record = dict(zip(COLUMNS, texts))
        # The Image column holds a picture, so store its URL instead of (empty) text.
        img = await cells[1].query_selector("img")
        record["Image"] = (await img.get_attribute("src")) if img else ""
        page_records.append(record)
    return page_records


async def scrape_all_pages():
    global page
    all_records = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(URL, wait_until="networkidle")

        for page_number in range(1, TOTAL_PAGES + 1):
            await page.wait_for_selector(ROW_SELECTOR)
            first_row_text = await page.inner_text(ROW_SELECTOR + ":first-child")

            page_records = await read_current_page()
            all_records.extend(page_records)
            print(f"Page {page_number:>2}/{TOTAL_PAGES}: {len(page_records)} rows "
                  f"(total so far: {len(all_records)})")

            if page_number == TOTAL_PAGES:
                break

            # Go to the next page (ASSUMPTION: a button/link whose text contains "Next").
            next_button = page.get_by_role("button", name=re.compile("next", re.I))
            if await next_button.count() == 0:
                next_button = page.get_by_role("link", name=re.compile("next", re.I))
            await next_button.first.click()

            # Wait until the first row changes, so we never read the same page twice.
            await page.wait_for_function(
                "(prev) => { const r = document.querySelector('table tbody tr');"
                " return r && r.innerText.trim() !== prev; }",
                arg=first_row_text, timeout=15000)

        await browser.close()
    return all_records


records = await scrape_all_pages()
print("\nTotal records scraped:", len(records))

## 4. Create the DataFrame

In [ ]:
df = pd.DataFrame(records, columns=COLUMNS)
print("Shape:", df.shape)
df.head()

## 5. Data validation (missing values and duplicates)

In [ ]:
print("Missing values per column:")
print(df.replace("", pd.NA).isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDuplicate IDs :", df["ID"].duplicated().sum())

## 6. Data cleaning

* Remove exact duplicate rows.
* Convert `Price`, `Rating` and `Variants` to numbers. Symbols such as `$` or `,` are stripped first.
  Values that cannot be converted become `NaN` (instead of crashing) and are counted below.

In [ ]:
def to_number(series):
    """Strip currency symbols/text, then convert to numbers. Bad values become NaN."""
    cleaned = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

df = df.drop_duplicates().reset_index(drop=True)

for column in ["Price", "Rating", "Variants"]:
    before_missing = df[column].isna().sum()
    df[column] = to_number(df[column])
    print(f"{column:<9} -> numeric | invalid/missing values: {df[column].isna().sum()}")

print("\nRecords after cleaning:", len(df))
print(df.dtypes)

In [ ]:
# Sanity check against the project scope (about 500 records over 20 pages).
if not 450 <= len(df) <= 550:
    print("WARNING: expected roughly 500 records but found", len(df),
          "- check the selectors and the pagination logic in Section 3.")
else:
    print("Record count is in the expected range.")

## 7. Save the cleaned dataset

In [ ]:
df.to_csv(CSV_PATH, index=False)
df.to_excel(XLSX_PATH, index=False)
print("Saved:", CSV_PATH)
print("Saved:", XLSX_PATH)

In [ ]:
# Colab only: download the files because the Colab disk is erased when the session ends.
try:
    from google.colab import files
    files.download(str(CSV_PATH))
    files.download(str(XLSX_PATH))
except ImportError:
    print("Not running in Colab - files are in the data/ folder.")

**Next step:** open `analysis/product_price_analysis.ipynb`. It reads `data/scraped_product_data.csv`.